## Income Classifier Inference

This notebook demonstrates how to load the trained income classifier, preprocess user-supplied attributes, and generate a predicted income bracket.

In [ ]:
import pandas as pd
import joblib
import numpy as np

# Load cleaned data
data = pd.read_csv('../data/cleaned.csv')

# Load the saved best model (XGBoost or any other model)
best_model = joblib.load('../models/best_xgboost_model.pkl')

# Load the scalers for preprocessing
hours_per_week_scaler = joblib.load('../tools/hours_scaler.pkl')
education_encoder = joblib.load('../tools/education_encoder.pkl')

# Function to preprocess user input with Label Encoding for 'education' and One-Hot Encoding for others
def preprocess_input(user_input):
    # Label encode 'education' column (since it is label encoded)
    user_input['education'] = education_encoder.transform([user_input['education']])[0]

    # Map user-friendly race input to dataset's race values
    race_value = user_input['race'].iloc[0]
    user_input['race'] = race_mapping.get(race_value, 'Other')  # Default to 'Other' if the input doesn't match

    # One-Hot Encode categorical variables (e.g., 'gender', 'workclass', 'marital-status', etc.)
    user_input = pd.get_dummies(user_input, columns=['gender', 'workclass', 'marital-status', 'occupation', 'relationship', 'race', 'native-country'])
    
    # Scale 'hours-per-week' field
    user_input['hours-per-week'] = hours_per_week_scaler.transform(user_input[['hours-per-week']])

    # Log scaling for 'capital-gain' and 'capital-loss'
    user_input['capital-gain'] = np.log1p(user_input['capital-gain'])  # log(x + 1) to avoid log(0)
    user_input['capital-loss'] = np.log1p(user_input['capital-loss'])  # log(x + 1) to avoid log(0)

    # Ensure all columns match the training data (add missing columns with 0s)
    required_columns = [col for col in data.columns if col != 'income']  # Get feature names from the model
    
    # Identify missing columns and add them with 0s
    missing_cols = set(required_columns) - set(user_input.columns)
    for col in missing_cols:
        user_input[col] = 0  # Add missing columns and set to 0
    
    # Reorder columns to match training data
    user_input = user_input[required_columns]
    
    return user_input